In [ ]:
# ==============================================================================
# FUNGSI CELL: Mengimpor library yang diperlukan dan menginisialisasi parameter 
# koneksi serta sertifikat SSL untuk Aiven Kafka.
# ==============================================================================

# Mengimpor modul bawaan Python 'os' untuk interaksi dengan sistem operasi (seperti path file)
import os

# Mengimpor library 'pandas' (sebagai pd) untuk analisis data dan manipulasi tabel DataFrame
import pandas as pd

# Mengimpor modul bawaan 'json' untuk memproses pertukaran data berformat JSON
import json

# Mengimpor modul bawaan 'time' untuk mengukur waktu proses dan menambahkan jeda (sleep)
import time

# Mengimpor modul bawaan 're' untuk pemrosesan teks menggunakan Regular Expression (Regex)
import re

# Mengimpor library 'numpy' (sebagai np) untuk operasi matematika dan representasi array/NaN
import numpy as np

# Mengimpor kelas produsen, klien administrator, dan partisi topik dari library 'kafka-python'
from kafka import KafkaProducer, KafkaAdminClient, TopicPartition

# Mengimpor kelas NewTopic untuk mendefinisikan topik baru yang akan dibuat di Kafka
from kafka.admin import NewTopic

# Menyimpan alamat broker Kafka Aiven Cloud dalam variabel KAFKA_BROKER
KAFKA_BROKER = "kafka-90a3cd4-cejors-676945.g.aivencloud.com:28174"

# Menyimpan nama topik Kafka yang digunakan untuk data pekerjaan gabungan
TOPIC_NAME = "unified_jobs"

# Mendefinisikan nama folder tempat penyimpanan sertifikat SSL
SSL_FOLDER = "ssl"

# Menggabungkan path folder SSL dengan nama file sertifikat otoritas (CA)
CA_FILE = os.path.join(SSL_FOLDER, "ca.pem")

# Menggabungkan path folder SSL dengan nama file sertifikat layanan (service certificate)
CERT_FILE = os.path.join(SSL_FOLDER, "service.cert")

# Menggabungkan path folder SSL dengan nama file kunci privat layanan (service key)
KEY_FILE = os.path.join(SSL_FOLDER, "service.key")

# Melakukan perulangan untuk memeriksa keberadaan masing-masing file sertifikat SSL
for f in [CA_FILE, CERT_FILE, KEY_FILE]:
    # Menggunakan os.path.exists untuk memverifikasi apakah file sertifikat tersebut ada di sistem
    if os.path.exists(f):
        # Mencetak konfirmasi jika file sertifikat SSL ditemukan
        print(f"  [OK] {f}")
    else:
        # Mencetak pesan kegagalan jika ada file sertifikat SSL yang tidak ditemukan
        print(f"  [GAGAL] {f} TIDAK DITEMUKAN!")

# Mencetak pesan bahwa semua library dan konfigurasi siap digunakan
print("\nLibrary dan parameter koneksi siap.")


  [OK] ssl/ca.pem
  [OK] ssl/service.cert
  [OK] ssl/service.key

Library dan parameter koneksi siap.


In [25]:
# ==============================================================================
# FUNGSI CELL: Menguji koneksi awal ke broker Aiven Kafka Cloud menggunakan 
# KafkaAdminClient dengan enkripsi SSL dan menampilkan daftar topik yang aktif.
# ==============================================================================

try:
  # Membuat objek KafkaAdminClient untuk melakukan operasi administratif pada Kafka
  admin_client = KafkaAdminClient(
      bootstrap_servers=KAFKA_BROKER, # Parameter bootstrap_servers: Menentukan alamat broker Kafka (host:port) untuk koneksi awal
      security_protocol='SSL',        # Parameter security_protocol: Protokol keamanan yang digunakan, di sini 'SSL' untuk koneksi aman terenkripsi
      ssl_cafile=CA_FILE,             # Parameter ssl_cafile: Path file sertifikat CA untuk memvalidasi identitas server
      ssl_certfile=CERT_FILE,         # Parameter ssl_certfile: Path file sertifikat client untuk otentikasi client di server
      ssl_keyfile=KEY_FILE            # Parameter ssl_keyfile: Path file private key client untuk enkripsi data otentikasi
  )

  # Mengambil daftar topik yang terdaftar di broker Kafka menggunakan fungsi list_topics()
  existing_topics = admin_client.list_topics()
  
  # Mencetak pesan sukses apabila koneksi berhasil dibentuk
  print(f"koneksi ke Aiven kafka berhasil")
  
  # Menampilkan daftar seluruh topik aktif saat ini di broker
  print(f"Topik yang aktif saatini: {existing_topics}")
  
  # Menutup koneksi KafkaAdminClient untuk membersihkan resource koneksi socket
  admin_client.close()

except Exception as e:
  # Menangkap error jika koneksi gagal dan mencetak pesan kesalahannya
  print(f"Gagal terhubung ke Aiven Kafka: {e}")
  
  # Memberikan petunjuk kepada pengguna untuk memastikan server Kafka aktif dan sertifikat benar
  print("pastikan service Aiven Anda sedang running dan sertifikat SSL sudah benar.")


koneksi ke Aiven kafka berhasil
Topik yang aktif saatini: ['unified_jobs', '__consumer_offsets']


In [26]:
# ==============================================================================
# FUNGSI CELL: Menghapus topik lama bernama 'unified_jobs' dari broker Aiven Kafka
# jika topik tersebut sudah ada, agar data streaming yang baru bersih dari data lama.
# ==============================================================================

def delete_old_topic():
  # Membuat objek KafkaAdminClient dengan konfigurasi koneksi SSL
  admin_client = KafkaAdminClient(
      bootstrap_servers=KAFKA_BROKER, # Parameter bootstrap_servers: Menentukan alamat broker Kafka (host:port) untuk koneksi awal
      security_protocol="SSL",        # Parameter security_protocol: Protokol keamanan yang digunakan, di sini 'SSL' untuk koneksi aman terenkripsi
      ssl_cafile=CA_FILE,             # Parameter ssl_cafile: Path file sertifikat CA untuk memvalidasi identitas server
      ssl_certfile=CERT_FILE,         # Parameter ssl_certfile: Path file sertifikat client untuk otentikasi client di server
      ssl_keyfile=KEY_FILE            # Parameter ssl_keyfile: Path file private key client untuk enkripsi data otentikasi
  )

  # Mengambil daftar topik saat ini dari broker Kafka
  existing_topics = admin_client.list_topics()
  print(f"Topik yang ada saat ini: {existing_topics}")

  # Memeriksa apakah topik 'unified_jobs' ada dalam daftar topik aktif
  if TOPIC_NAME in existing_topics:
    print(f"Menghapus topik '{TOPIC_NAME}'...")
    # Menghapus topik dari broker menggunakan fungsi delete_topics()
    # Parameter delete_topics: Menerima list berisi nama topik yang akan dihapus di broker
    admin_client.delete_topics([TOPIC_NAME])
    print(f"Topik '{TOPIC_NAME}' berhasil dihapus.")
    print("Menunggu 5 detik agar broker menyelesaikan penghapusan...")
    # Menggunakan time.sleep untuk menunda eksekusi agar broker menyelesaikan proses penghapusan
    # Parameter sleep: Menerima nilai integer berupa durasi penundaan dalam detik (5 detik)
    time.sleep(5)
  else:
    print(f"Topik '{TOPIC_NAME}' tidak ditemukan, tidak perlu dihapus.")

  # Menutup objek KafkaAdminClient
  admin_client.close()

# Memanggil fungsi delete_old_topic() untuk menghapus topik yang ada
delete_old_topic()


Topik yang ada saat ini: ['unified_jobs', '__consumer_offsets']
Menghapus topik 'unified_jobs'...
Topik 'unified_jobs' berhasil dihapus.
Menunggu 5 detik agar broker menyelesaikan penghapusan...


In [27]:
# ==============================================================================
# FUNGSI CELL: Membuat topik baru bernama 'unified_jobs' dengan 1 partisi dan 
# faktor replikasi 1 jika topik tersebut belum terbuat di broker Aiven Kafka.
# ==============================================================================

def create_fresh_topic():
  # Membuat objek KafkaAdminClient dengan koneksi SSL untuk kebutuhan administrasi topik
  admin_client = KafkaAdminClient(
      bootstrap_servers=KAFKA_BROKER, # Parameter bootstrap_servers: Menentukan alamat broker Kafka (host:port) untuk koneksi awal
      security_protocol="SSL",        # Parameter security_protocol: Protokol keamanan yang digunakan, di sini 'SSL' untuk koneksi aman terenkripsi
      ssl_cafile=CA_FILE,             # Parameter ssl_cafile: Path file sertifikat CA untuk memvalidasi identitas server
      ssl_certfile=CERT_FILE,         # Parameter ssl_certfile: Path file sertifikat client untuk otentikasi client di server
      ssl_keyfile=KEY_FILE            # Parameter ssl_keyfile: Path file private key client untuk enkripsi data otentikasi
  )

  # Mendapatkan list topik yang saat ini terdaftar di broker
  existing_topics = admin_client.list_topics()

  # Memeriksa jika topik target (unified_jobs) belum ada dalam broker
  if TOPIC_NAME not in existing_topics:
    print(f"Membuat topik baru '{TOPIC_NAME}'...")
    # Membuat objek NewTopic untuk mendefinisikan topik baru yang akan dibuat di broker Kafka
    topic = NewTopic(
        name=TOPIC_NAME,            # Parameter name: Nama topik yang akan dibuat di Kafka ('unified_jobs')
        num_partitions=1,           # Parameter num_partitions: Jumlah partisi untuk distribusi data dan skalabilitas paralel (di sini 1 partisi)
        replication_factor=1        # Parameter replication_factor: Jumlah salinan data di broker untuk ketahanan (di sini 1 replika)
    )
    # Memproses pembuatan topik baru ke broker menggunakan admin_client.create_topics()
    # Parameter new_topics: List objek NewTopic yang akan dikirim ke broker untuk dibuat
    # Parameter validate_only: Boolean untuk memeriksa kevalidan konfigurasi tanpa membuat topiknya (False = langsung buat)
    admin_client.create_topics(new_topics=[topic], validate_only=False)
    print(f"Topik '{TOPIC_NAME}' berhasil dibuat!")
  else:
    print(f"Topik '{TOPIC_NAME}' sudah ada dan siap digunakan.")

  # Menutup client admin Kafka
  admin_client.close()

# Menjalankan fungsi pembuatan topik baru
create_fresh_topic()


Membuat topik baru 'unified_jobs'...
Topik 'unified_jobs' berhasil dibuat!


In [28]:
# ==============================================================================
# FUNGSI CELL: Membaca dataset lowongan kerja dari Adzuna API (format CSV),
# menampilkan dimensi baris/kolom, serta mendeteksi kolom yang memiliki nilai kosong (Null).
# ==============================================================================

# Mencetak garis pemisah estetis menggunakan perkalian string
print("=" * 60)
# Mencetak judul log proses Extraction untuk dataset Adzuna API
print("[Extraction] Membaca Dataset Adzuna Api")
print("=" * 60)

# Menetapkan lokasi path file CSV Adzuna API ke variabel adzuna_csv
adzuna_csv = "Adzuna API/adzuna_jobs.csv"

# Membaca file CSV tersebut ke objek DataFrame pandas df_adz menggunakan pd.read_csv()
df_adz = pd.read_csv(
    adzuna_csv # Parameter pertama: Path atau nama file CSV yang akan dibaca ke dalam DataFrame
)

# Menampilkan informasi path file yang sedang dibaca
print(f"File  : {adzuna_csv}")
# Menampilkan jumlah baris dengan format pemisah ribuan koma memakai len()
print(f"Baris : {len(df_adz):,}")
# Menampilkan daftar kolom yang tersedia menggunakan df_adz.columns
print(f"Kolom : {list(df_adz.columns)}")
print(f"Null  :")
# Menghitung jumlah nilai null per kolom, lalu memfilter hanya kolom yang memiliki null > 0
print(df_adz.isnull().sum()[df_adz.isnull().sum() > 0])
print()
# Menampilkan 3 baris teratas (sampel data) dari DataFrame Adzuna API
df_adz.head(3)


[Extraction] Membaca Dataset Adzuna Api
File  : Adzuna API/adzuna_jobs.csv
Baris : 280
Kolom : ['job_id', 'title', 'description', 'company_name', 'location', 'location_area', 'min_salary', 'max_salary', 'salary_is_predicted', 'contract_time', 'contract_type', 'created_time', 'category_tag', 'category_label', 'search_role', 'redirect_url']
Null  :
contract_time    218
contract_type    245
dtype: int64



,job_id,title,description,company_name,location,location_area,min_salary,max_salary,salary_is_predicted,contract_time,contract_type,created_time,category_tag,category_label,search_role,redirect_url
0,5712067843,Data Engineer,ManTech seeks a Data Engineer to support our I...,ManTech International,"Chantilly, Fairfax County","US, Virginia, Fairfax County, Chantilly",115000.00,160000.00,0,NaN,NaN,2026-04-27T19:15:00Z,it-jobs,IT Jobs,Data Engineer,https://www.adzuna.com/land/ad/5712067843?se=a...
1,5712067907,Senior Data Engineer,ManTech International seeks a Senior Data Engi...,ManTech International,"Chantilly, Fairfax County","US, Virginia, Fairfax County, Chantilly",145000.00,180000.00,0,NaN,NaN,2026-04-27T19:15:01Z,it-jobs,IT Jobs,Data Engineer,https://www.adzuna.com/land/ad/5712067907?se=a...
2,5728506923,"Principal, Data Engineering",Cargill is committed to providing food and agr...,Cargill,"Atlanta, Fulton County","US, Georgia, Fulton County, Atlanta",145712.25,145712.25,1,NaN,NaN,2026-05-13T17:43:28Z,it-jobs,IT Jobs,Data Engineer,https://www.adzuna.com/land/ad/5728506923?se=a...


In [29]:
# ==============================================================================
# FUNGSI CELL: Membaca dataset lowongan kerja Kaggle LinkedIn (postings.csv),
# menampilkan statistika jumlah baris/kolom, serta menganalisis persentase nilai kosong
# dan menghitung jumlah duplikasi data pada ID/Judul lowongan.
# ==============================================================================

print("=" * 60)
# Mencetak judul log proses Extraction untuk dataset LinkedIn postings
print("[EXTRACTION] Membaca Dataset Kaggle LinkedIn (TANPA LIMIT)")
print("=" * 60)

# Menetapkan lokasi path file CSV LinkedIn postings
kaggle_csv = "kaggle - scaraping job from linkedin 2023 - 2024/postings.csv"

# Membaca dataset postings.csv ke DataFrame pandas df_kag
df_kag = pd.read_csv(
    kaggle_csv # Parameter pertama: Path atau nama file CSV yang akan dibaca ke dalam DataFrame
)

print(f"File  : {kaggle_csv}")
# Menampilkan total baris dalam dataset Kaggle LinkedIn
print(f"baris : {len(df_kag):,}")
# Menampilkan jumlah kolom dalam dataset Kaggle LinkedIn
print(f"Kolom : {len(df_kag.columns)} kolom")
print(f"\nStatistik Null (kolom yang akan kita gunakan): ")

# Mendefinisikan kolom-kolom penting yang akan diekstrak dan digunakan dalam pipeline
col_interest = ['job_id', 'title', 'description', 'company_name', 'location', 'min_salary', 'max_salary', 'work_type', 'formatted_experience_level', 'skills_desc']

# Melakukan perulangan untuk menghitung nilai null pada setiap kolom terpilih
for col in col_interest:
  # Menghitung jumlah nilai kosong pada kolom menggunakan isnull().sum()
  null_count = df_kag[col].isnull().sum()
  # Menghitung persentase nilai kosong terhadap total baris data bersih
  pct = (null_count / len(df_kag)) * 100
  # Mencetak hasil dengan format rata kiri/kanan agar rapi
  print(f" {col:35s} -> {null_count:>6,} null ({pct:.1f}%)")

# Menhitung jumlah duplikat berdasarkan kolom kunci 'job_id'
print(f"\nDuplikat job_id   : {df_kag['job_id'].duplicated().sum():,}")
# Menghitung jumlah baris yang memiliki kesamaan judul pekerjaan dan deskripsi (spam lowongan)
# Parameter subset: List nama kolom untuk mendefinisikan kombinasi keunikan duplikasi baris
print(f"Duplikat title+desc : {df_kag.duplicated(subset=['title','description']).sum():,}")
print()
# Menampilkan 3 baris sampel teratas dari data LinkedIn
df_kag.head(3)


[EXTRACTION] Membaca Dataset Kaggle LinkedIn (TANPA LIMIT)
File  : kaggle - scaraping job from linkedin 2023 - 2024/postings.csv
baris : 123,849
Kolom : 31 kolom

Statistik Null (kolom yang akan kita gunakan): 
 job_id                              ->      0 null (0.0%)
 title                               ->      0 null (0.0%)
 description                         ->      7 null (0.0%)
 company_name                        ->  1,719 null (1.4%)
 location                            ->      0 null (0.0%)
 min_salary                          -> 94,056 null (75.9%)
 max_salary                          -> 94,056 null (75.9%)
 work_type                           ->      0 null (0.0%)
 formatted_experience_level          -> 29,409 null (23.7%)
 skills_desc                         -> 121,410 null (98.0%)

Duplikat job_id   : 0
Duplikat title+desc : 12,944



,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


In [30]:
# ==============================================================================
# FUNGSI CELL: Membaca dataset standar profesi O*NET (Occupation Data dan Technology Skills),
# menggabungkan (agregasi) daftar skill teknologi berdasarkan kode profesi (O*NET-SOC Code) 
# menjadi bentuk list teks terpisah koma.
# ==============================================================================

print("="*60)
# Log proses Extraction untuk data referensi O*NET
print("[EXTRACTION] Membaca dataset O*net")
print("="*60)

# Menentukan path file Excel data Profesi O*NET
occ_path = "ONET Occupational Database/db_30_2_excel/Occupation Data.xlsx"
# Menentukan path file Excel data Skill Teknologi O*NET
tech_path = "ONET Occupational Database/db_30_2_excel/Technology Skills.xlsx"

# Membaca file Excel data profesi ke DataFrame df_occ menggunakan pd.read_excel()
# Parameter pertama: Path atau nama file Excel (.xlsx) yang akan dibaca
df_occ = pd.read_excel(occ_path)
# Membaca file Excel data skill ke DataFrame df_tech
df_tech = pd.read_excel(tech_path)

print(f"\nOccupation Data:")
print(f"  Baris  : {len(df_occ):,} profesi")
print(f"  Kolom  : {list(df_occ.columns)}")
print(f"\nTechnology Skills:")
print(f"  Baris  : {len(df_tech):,} entri skill")
print(f"  Kolom  : {list(df_tech.columns)}")

# Preview pengelompokan skill per profesi:
# Mengelompokkan df_tech berdasarkan 'O*NET-SOC Code' dan menggabungkan nilai kolom 'Example' 
# yang unik dipisahkan dengan tanda koma (', ')
# Parameter pertama dari groupby: Nama kolom atau list kolom yang digunakan sebagai kunci pengelompokan
tech_grouped = df_tech.groupby('O*NET-SOC Code')['Example'].apply(
    lambda x: ', '.join(x.astype(str).unique())
).reset_index()
# Mengubah nama kolom pada DataFrame hasil agregasi
tech_grouped.columns = ['O*NET-SOC Code', 'skills_aggregated']

print(f"\nSetelah agregasi, setiap profesi memiliki daftar skill gabungan.")
print(f"Contoh:")
# Menampilkan contoh 2 baris teratas hasil pengelompokan tanpa indeks dataframe
# Parameter index: Boolean untuk menampilkan/menyembunyikan kolom index baris dataframe (False = sembunyikan)
print(tech_grouped.head(2).to_string(index=False))


[EXTRACTION] Membaca dataset O*net

Occupation Data:
  Baris  : 1,016 profesi
  Kolom  : ['O*NET-SOC Code', 'Title', 'Description']

Technology Skills:
  Baris  : 32,773 entri skill
  Kolom  : ['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code', 'Commodity Title', 'Hot Technology', 'In Demand']

Setelah agregasi, setiap profesi memiliki daftar skill gabungan.
Contoh:
O*NET-SOC Code                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [31]:
# ==============================================================================
# FUNGSI CELL: Menghasilkan ringkasan total data mentah (*raw data*) yang berhasil 
# diekstrak dari semua sumber data (Adzuna, LinkedIn, O*NET).
# ==============================================================================

print("=" * 60)
print("RINGKASAN EXTRACTION")
print("=" * 60)
# Menampilkan jumlah baris masing-masing dataset dengan format rata kanan 10 digit (:>10,)
print(f"  Adzuna API         : {len(df_adz):>10,} baris")
print(f"  Kaggle LinkedIn    : {len(df_kag):>10,} baris")
print(f"  O*NET Occupations  : {len(df_occ):>10,} profesi")
print(f"  O*NET Tech Skills  : {len(df_tech):>10,} entri skill")
print(f"  {'':->40}")

# Menjumlahkan seluruh baris data lowongan kerja mentah dari Adzuna, Kaggle, dan O*NET
total_raw = len(df_adz) + len(df_kag) + len(df_occ)
# Menampilkan total keseluruhan data mentah yang telah dimuat
print(f"  TOTAL DATA MENTAH  : {total_raw:>10,} baris")
print(f"\nData mentah siap untuk masuk ke tahap Transformation & Cleaning.")


RINGKASAN EXTRACTION
  Adzuna API         :        280 baris
  Kaggle LinkedIn    :    123,849 baris
  O*NET Occupations  :      1,016 profesi
  O*NET Tech Skills  :     32,773 entri skill
  ----------------------------------------
  TOTAL DATA MENTAH  :    125,145 baris

Data mentah siap untuk masuk ke tahap Transformation & Cleaning.


In [32]:
# ==============================================================================
# FUNGSI CELL: Memetakan (mapping) struktur kolom dataset Adzuna API ke dalam
# skema data standar (Standard Unified Schema) agar seragam dengan dataset lainnya.
# ==============================================================================

print("[TRANSFORM] Memetakan data adzuna ke format standar...")

# Membuat list kosong records_adzuna untuk menampung dictionary data hasil transformasi
records_adzuna = []

# Iterasi baris per baris dari DataFrame Adzuna df_adz menggunakan iterrows()
for idx, row in df_adz.iterrows():
    # Menambahkan data berbentuk dictionary terstandardisasi ke dalam list records_adzuna
    # Menggunakan fungsi strip() untuk membersihkan spasi di awal/akhir string, dan pd.notna() 
    # untuk mengecek nilai kosong
    # Parameter pd.notna: Menerima objek/nilai yang akan diperiksa (mengembalikan True jika tidak bernilai kosong/NaN)
    records_adzuna.append({
        'unique_id': f"ADZ_{row['job_id']}", # Membuat ID unik dengan prefix ADZ_
        'data_source': 'Adzuna_API',         # Menandai sumber data asal
        'job_title': str(row['title']).strip(),
        'company_name': str(row['company_name']).strip() if pd.notna(row['company_name']) else None,
        # Mengambil maksimal 2000 karakter dari deskripsi pekerjaan untuk efisiensi performa
        'job_description': str(row['description'])[:2000].strip() if pd.notna(row['description']) else None,
        'location': str(row['location']).strip() if pd.notna(row['location']) else None,
        # Mengkonversi gaji ke tipe float
        'min_salary': float(row['min_salary']) if pd.notna(row['min_salary']) else None,
        'max_salary': float(row['max_salary']) if pd.notna(row['max_salary']) else None,
        'contract_time': str(row['contract_time']).strip() if pd.notna(row['contract_time']) else None,
        'contract_type': str(row['contract_type']).strip() if pd.notna(row['contract_type']) else None,
        'experience_level': 'Not Specified', # Default value karena Adzuna tidak menyediakan level pengalaman
        'skills_required': None              # Default value
    })

# Menampilkan jumlah record Adzuna API yang telah berhasil ditransformasikan
print(f"  Adzuna berhasil dipetakan: {len(records_adzuna):,} baris")


[TRANSFORM] Memetakan data adzuna ke format standar...
  Adzuna berhasil dipetakan: 280 baris


In [33]:
# ==============================================================================
# FUNGSI CELL: Memetakan (mapping) kolom dataset LinkedIn (Kaggle postings) ke 
# dalam skema data standar, mengeliminasi lowongan tanpa judul/deskripsi, dan 
# mencetak log progres setiap 25.000 baris.
# ==============================================================================

print("[TRANSFORM] Memetakan data Kaggle LinkedIn ke format standar...")
print("  (Proses ini memakan waktu beberapa detik karena dataset besar)")

# List kosong records_kaggle untuk menampung dictionary data LinkedIn hasil pemetaan
records_kaggle = []

# Iterasi baris per baris pada DataFrame LinkedIn df_kag
for idx, row in df_kag.iterrows():
    # Filter validasi awal: menggunakan pd.isna() untuk melewati data yang tidak memiliki judul atau deskripsi
    # Parameter pd.isna: Menerima nilai atau ekspresi (mengembalikan True jika nilainya kosong/NaN)
    if pd.isna(row['title']) or pd.isna(row['description']):
        continue

    # Menyusun kamus data terstandardisasi
    records_kaggle.append({
        'unique_id': f"KAG_{row['job_id']}", # ID unik dengan prefix KAG_
        'data_source': 'Kaggle_LinkedIn',
        'job_title': str(row['title']).strip(),
        'company_name': str(row['company_name']).strip() if pd.notna(row['company_name']) else None,
        'job_description': str(row['description'])[:2000].strip() if pd.notna(row['description']) else None,
        'location': str(row['location']).strip() if pd.notna(row['location']) else None,
        'min_salary': float(row['min_salary']) if pd.notna(row['min_salary']) else None,
        'max_salary': float(row['max_salary']) if pd.notna(row['max_salary']) else None,
        'contract_time': str(row['work_type']).strip() if pd.notna(row['work_type']) else None,
        'contract_type': 'Not Specified',
        'experience_level': str(row['formatted_experience_level']).strip() if pd.notna(row['formatted_experience_level']) else 'Not Specified',
        'skills_required': str(row['skills_desc'])[:1000].strip() if pd.notna(row['skills_desc']) else None
    })

    # Log progres agar proses eksekusi terpantau setiap kelipatan 25.000 data
    if (idx + 1) % 25000 == 0:
        print(f"  ... {idx + 1:,} baris diproses")

# Menampilkan total data LinkedIn postings yang berhasil dipetakan
print(f"  Kaggle berhasil dipetakan: {len(records_kaggle):,} baris")


[TRANSFORM] Memetakan data Kaggle LinkedIn ke format standar...
  (Proses ini memakan waktu beberapa detik karena dataset besar)
  ... 25,000 baris diproses
  ... 50,000 baris diproses
  ... 75,000 baris diproses
  ... 100,000 baris diproses
  Kaggle berhasil dipetakan: 123,842 baris


In [34]:
# ==============================================================================
# FUNGSI CELL: Menggabungkan (join) data profesi O*NET dengan data skill teknologi 
# hasil agregasi, lalu memetakan hasilnya ke skema standar unified.
# ==============================================================================

print("[TRANSFORM] Memetakan data O*net ke format standar..")

# Menggabungkan DataFrame df_occ dan tech_grouped berdasarkan kolom key 'O*NET-SOC Code'
# Menggunakan pd.merge() dengan parameter how='left' (Left Join) agar semua data okupasi tetap ada
df_onet_merged = pd.merge(
    df_occ,           # Parameter pertama: DataFrame kiri yang akan digabungkan (data okupasi)
    tech_grouped,     # Parameter kedua: DataFrame kanan yang akan digabungkan (data skill teknologi okupasi)
    on='O*NET-SOC Code', # Parameter on: Kolom kunci utama yang digunakan sebagai dasar penggabungan data
    how='left'        # Parameter how: Metode join (di sini 'left' join agar seluruh baris df_occ tetap dipertahankan)
)

# List untuk menampung dictionary data standar O*NET
records_onet = []

# Melakukan perulangan baris per baris pada data gabungan O*NET
for idx, row in df_onet_merged.iterrows():
  records_onet.append({
      'unique_id': f"ONE_{row['O*NET-SOC Code']}", # ID unik dengan prefix ONE_
      'data_source': 'ONET_Standard',
      'job_title': str(row['Title']).strip(),
      'company_name': 'ONET Standard System',        # Nama perusahaan diset default sistem standar
      'job_description': str(row['Description'])[:2000].strip() if pd.notna(row['Description']) else None,
      'location': 'Global',                           # Default lokasi diset Global
      'min_salary': None,
      'max_salary': None,
      'contract_time': 'Standard',
      'contract_type': 'Standard',
      'experience_level': 'Standard',
      'skills_required': str(row['skills_aggregated'])[:1000] if pd.notna(row['skills_aggregated']) else None
  })

# Menampilkan total profesi O*NET yang berhasil ditransformasikan
print(f"O*NET berhasil dipetakan: {len(records_onet):,} baris")


[TRANSFORM] Memetakan data O*net ke format standar..
O*NET berhasil dipetakan: 1,016 baris


In [35]:
# ==============================================================================
# FUNGSI CELL: Menggabungkan list dictionary dari ketiga sumber data (Adzuna,
# LinkedIn, O*NET), mengonversinya ke dalam satu DataFrame tunggal, dan menampilkan 
# statistik komposisi datanya.
# ==============================================================================

# Menampilkan informasi dimulainya penggabungan data
print("[TRANSFORM] Menggabungkan seluruh data ...")

# Menggabungkan ketiga list record data (records_adzuna, records_kaggle, dan records_onet)
# menggunakan operator penambahan list python (+)
all_records = records_adzuna + records_kaggle + records_onet

# Mengonversi list dictionary all_records menjadi objek DataFrame pandas (df_unified)
# Parameter pertama: List objek kamus data pekerjaan yang akan dimasukkan ke DataFrame
df_unified = pd.DataFrame(all_records)

# Menampilkan jumlah baris data hasil penggabungan sebelum proses pembersihan
print(f"Total data gabungan (sebelum cleaning): {len(df_unified):,} baris")
print(f"\n Komposisi per sumber data:")
# Menghitung frekuensi data_source menggunakan value_counts() dan mencetaknya ke format string
print(df_unified['data_source'].value_counts().to_string())


[TRANSFORM] Menggabungkan seluruh data ...


Total data gabungan (sebelum cleaning): 125,138 baris

 Komposisi per sumber data:
data_source
Kaggle_LinkedIn    123842
ONET_Standard        1016
Adzuna_API            280


In [36]:
# ==============================================================================
# FUNGSI CELL: Menjalankan 7 tahapan pembersihan data (*data cleaning*) secara 
# menyeluruh meliputi penghapusan data kosong, penghapusan data duplikat, harmonisasi 
# sinonim lokasi, penghapusan spasi liar, dan penanganan nilai kosong untuk JSON.
# ==============================================================================

print("="*60)
print("[CLEANING] Memulai proses pembersihan data menyeluruh")
print("="*60)

# Menduplikasi DataFrame df_unified ke df_clean agar tidak merusak data awal menggunakan .copy()
df_clean = df_unified.copy()
print(f"\n [0] ukuran awal : {len(df_clean):>10,} baris")

# cleaning 1: hapus baris yang kolom job_title atau job_description bernilai null menggunakan dropna()
# Parameter subset: List nama kolom target yang akan diperiksa nilai null-nya
df_clean = df_clean.dropna(subset=['job_title', 'job_description'])
print(f" [1] setelah hapus null title/description : {len(df_clean):>10,} baris")

# cleaning 2: menghapus duplikat ID unik menggunakan drop_duplicates()
# Parameter subset: Kolom target acuan pengecekan nilai duplikat (unik)
df_clean = df_clean.drop_duplicates(subset=['unique_id'])
print(f" [2] setelah hapus duplikat unique_id : {len(df_clean):>10,} baris")

# cleaning 3: menghapus spam rekrutmen (job_title dan job_description yang kembar persis)
# Parameter subset: Kombinasi kolom acuan untuk mendeteksi duplikat
df_clean = df_clean.drop_duplicates(subset=['job_title', 'job_description'])
print(f" [3] setelah hapus spam rekruter : {len(df_clean):>10,} baris")

# cleaning 4: menstandardisasi penulisan contract_time menjadi huruf kapital menggunakan str.upper()
df_clean['contract_time'] = df_clean['contract_time'].str.upper()
unique_ct = df_clean['contract_time'].dropna().unique()
print(f" [4] contract_time dseragamkan : {len(df_clean):>10,} baris")

# cleaning 5: menyamakan penulisan nama lokasi yang memiliki makna sama (sinonim) dengan kamus loc_mapping
loc_mapping = {
    "US": "United States",
    "New York City Metropolitan Area": "New York, NY",
    "Austin, Texas Metropolitan Area": "Austin, TX",
    "San Fransisco Bay Area": "San Francisco, CA",
    "Greater Chicago Area": "Chicago, IL",
    "Greater Boston": "Boston, MA",
    "Dallas-Fort Worth Metroplex": "Dallas, TX",
    "Greater Seattle Area": "Seattle, WA",
    "Greater Los Angeles Area": "Los Angeles, CA",
    "Greater Philadelphia": "Philadelphia, PA",
    "Greater Denver Area": "Denver, CO",
    "Greeater Minneapolis-St. Paul Area": "Minneapolis, MN",
    "Washington DC-Baltimore Area": "Washington, DC",
    "Greater Tampa Bay Area": "Tampa, FL",
    "Greater Portland Oregon Area": "Portland, OR",
    "Greater Sacramento Area": "Sacramento, CA",
    "Greater Nashville Area, TN": "Nashville, TN",
}
# Menerapkan penggantian string lokasi berdasarkan pemetaan kamus menggunakan replace()
# Parameter pertama: Kamus pemetaan pengganti nilai (key diganti dengan value)
df_clean['location'] = df_clean['location'].replace(loc_mapping)
print(f" [5] Lokasi sinonim diharmonisasi : {len(loc_mapping)} Pemetaan diterapkan")

# cleaning 6: menghapus spasi di awal/akhir (*whitespace stripping*) pada kolom bertipe data object/string
# Parameter include: Tipe data target kolom yang akan diseleksi (di sini bertipe 'object' / teks)
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
  df_clean[col] = df_clean[col].str.strip()
print(f" [6] whitespace berlebihan dibersihkan : {len(str_cols)} kolom string")

# cleaning 7: mengganti tipe data kosong Pandas (np.nan) dengan objek None bawaan Python 
# agar kompatibel dan aman saat diserialisasi ke format JSON ketika dikirim ke Kafka
# Parameter pertama: Kamus pencarian penggantian nilai (np.nan diganti None)
df_clean = df_clean.replace({np.nan: None})
print(f" [7] NaN dikonversi ke None (JSON-safe)")

# Hasil AKhir setelah cleaning
print(f"\n{'=' * 60}")
print(f"Data bersih akhir: {len(df_clean):,} baris siap dikirim ke aiven")
print(f"{'=' * 60}")
print(f"\n Komposisi per sumber data:")
print(df_clean['data_source'].value_counts().to_string())


[CLEANING] Memulai proses pembersihan data menyeluruh

 [0] ukuran awal :    125,138 baris
 [1] setelah hapus null title/description :    125,138 baris
 [2] setelah hapus duplikat unique_id :    125,138 baris
 [3] setelah hapus spam rekruter :    108,963 baris
 [4] contract_time dseragamkan :    108,963 baris
 [5] Lokasi sinonim diharmonisasi : 17 Pemetaan diterapkan
 [6] whitespace berlebihan dibersihkan : 10 kolom string
 [7] NaN dikonversi ke None (JSON-safe)

Data bersih akhir: 108,963 baris siap dikirim ke aiven

 Komposisi per sumber data:
data_source
Kaggle_LinkedIn    107788
ONET_Standard        1016
Adzuna_API            159


/tmp/ipykernel_25463/2864179758.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df_clean.select_dtypes(include='object').columns


In [37]:
# ==============================================================================
# FUNGSI CELL: Menghasilkan Laporan Kualitas Data (*Data Quality Report*) setelah
# pembersihan, mendeteksi sisa null/duplikat, serta menampilkan sebaran 10 lokasi teratas.
# ==============================================================================

print("Laporan kualitas data bersih")

# Menghitung sisa data duplikat ID unik menggunakan df_clean.duplicated().sum()
# Parameter subset: Kolom target acuan pengecekan duplikasi
print(f"\n Sisa duplikat unique_id : {df_clean.duplicated(subset=['unique_id']).sum()}")
# Menghitung sisa data duplikat kombinasi judul dan deskripsi pekerjaan
print(f" Sisa duplikat title+desc : {df_clean.duplicated(subset=['job_title', 'job_description']).sum()}")

print(f"\n Statistik null per kolom:")
# Melakukan perulangan untuk memeriksa data null pada setiap kolom DataFrame bersih
for col in df_clean.columns:
  # Menghitung jumlah nilai null
  null_count = df_clean[col].isnull().sum()
  # Menghitung persentase null terhadap total baris data bersih
  pct = (null_count / len(df_clean)) * 100
  # Membuat status pelaporan teks untuk tiap kolom
  status = "Bersih" if null_count == 0 else f"{null_count:,} null ({pct:.1f}%)"
  print(f" {col:25s} -> {status}")

print(f"\n top 10 lokasi:")
# Menggunakan value_counts().head(10) untuk mengambil 10 besar lokasi terbanyak dan mencetaknya
# Parameter head: Nilai integer jumlah baris teratas yang ingin diambil (di sini 10 baris)
print(f" {df_clean['location'].value_counts().head(10).to_string()}")
print(f"\nData siap untuk distream ke aiven kafka")


Laporan kualitas data bersih

 Sisa duplikat unique_id : 0
 Sisa duplikat title+desc : 0

 Statistik null per kolom:
 unique_id                 -> Bersih
 data_source               -> Bersih
 job_title                 -> Bersih
 company_name              -> 1,698 null (1.6%)
 job_description           -> Bersih
 location                  -> Bersih
 min_salary                -> 81,027 null (74.4%)
 max_salary                -> 81,027 null (74.4%)
 contract_time             -> 156 null (0.1%)
 contract_type             -> 157 null (0.1%)
 experience_level          -> Bersih
 skills_required           -> 105,966 null (97.2%)

 top 10 lokasi:
 location
United States     7761
New York, NY      3358
Chicago, IL       1855
Houston, TX       1556
Dallas, TX        1471
Atlanta, GA       1244
Austin, TX        1210
Boston, MA        1158
Washington, DC    1057
Global            1016

Data siap untuk distream ke aiven kafka


In [38]:
# ==============================================================================
# FUNGSI CELL: Menginisialisasi objek KafkaProducer dengan opsi enkripsi SSL,
# serialisasi JSON, penanganan retries, buffering batch (linger/batch size) 
# untuk transmisi data yang efisien ke Aiven Kafka.
# ==============================================================================

print("[Kafka] menginisialisasi producer dengan enkripsi ssl...")

# Membuat objek produsen Kafka dengan parameter transmisi berkinerja tinggi
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER, # Parameter bootstrap_servers: Alamat server broker Kafka (host:port)
    security_protocol="SSL",        # Parameter security_protocol: Protokol keamanan (SSL) untuk mengenkripsi pengiriman data
    ssl_cafile=CA_FILE,             # Parameter ssl_cafile: Sertifikat CA tepercaya untuk validasi sertifikat server Aiven
    ssl_certfile=CERT_FILE,         # Parameter ssl_certfile: Sertifikat client untuk otentikasi
    ssl_keyfile=KEY_FILE,           # Parameter ssl_keyfile: Private key client untuk proses enkripsi
    # Parameter value_serializer: Fungsi pengubah format data (di sini mengonversi dictionary ke string JSON lalu ke byte)
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    request_timeout_ms=300000,      # Parameter request_timeout_ms: Waktu batas tunggu respon pengiriman (5 menit / 300.000 ms)
    max_block_ms=300000,            # Parameter max_block_ms: Batas waktu maksimal memblokir pengiriman saat buffer penuh (5 menit)
    retries=10,                     # Parameter retries: Jumlah percobaan ulang otomatis jika pengiriman gagal akibat gangguan jaringan (10 kali)
    linger_ms=100,                  # Parameter linger_ms: Waktu tunda buffering pesan (100 ms) sebelum dikirim secara massal (batching)
    batch_size=32768                # Parameter batch_size: Batas maksimum ukuran memori per batch pengiriman (32 KB / 32.768 byte)
)

print("producer berhasil terhubung dengan aiven kafka cloud")
print(f"target: {len(df_clean):,} pesan akan dikirim ke topik '{TOPIC_NAME}'")


[Kafka] menginisialisasi producer dengan enkripsi ssl...
producer berhasil terhubung dengan aiven kafka cloud
target: 108,963 pesan akan dikirim ke topik 'unified_jobs'


In [39]:
# ==============================================================================
# FUNGSI CELL: Mengubah DataFrame bersih menjadi list dictionary, lalu melakukan
# streaming data baris demi baris ke broker Aiven Kafka dengan pelacakan progres 
# real-time, jeda anti-limit, dan pembilasan (*flushing*) buffer akhir.
# ==============================================================================

print(f"Memulai streaming {len(df_clean):,} data bersih ke aiven kafka")

# Mengonversi baris DataFrame df_clean menjadi list objek dictionary memakai to_dict()
# Parameter orient: Format penyusunan struktur kamus data ('records' = list dari baris-baris berformat dict)
records_to_send = df_clean.to_dict(orient='records')

# Inisialisasi variabel penghitung pesan sukses dan gagal serta waktu mulai
success_count = 0
fail_count = 0
start_time = time.time() # Mengambil timestamp detik saat ini untuk pencatatan durasi

# Mengiterasi setiap record data pekerjaan untuk dikirimkan
for idx, record in enumerate(records_to_send):
  try:
    # Mengirimkan pesan data secara asinkron ke topik Kafka menggunakan producer.send()
    # Parameter pertama: Nama topik Kafka tujuan pengiriman pesan ('unified_jobs')
    # Parameter value: Payload data yang dikirim (berupa dictionary data pekerjaan)
    future = producer.send(TOPIC_NAME, value=record)
    # Menunggu respon keberhasilan penerimaan dari broker Kafka
    # Parameter timeout: Batas waktu tunggu maksimal dalam detik sebelum memicu error timeout (10 detik)
    future.get(timeout=10)
    success_count += 1

    # Jeda pengaman anti-limit (10 milidetik per pesan) agar tidak membebani broker Aiven Kafka secara berlebih
    # Parameter sleep: Durasi waktu tunda eksekusi program dalam detik (0.01 detik = 10 ms)
    time.sleep(0.01)

    # Cetak laporan progres setiap kelipatan 1.000 pesan sukses terkirim
    if success_count % 1000 == 0:
      # Menghitung selang waktu berjalan dalam detik
      elapsed = time.time() - start_time
      # Menghitung rata-rata kecepatan pengiriman pesan per detik (speed)
      speed = success_count / elapsed
      # Menghitung estimasi sisa waktu pengiriman (remaining) dalam detik
      remaining = (len(records_to_send) - success_count) / speed
      print(f" [{success_count:>7,} / {len(records_to_send):,}] "
            f"Waktu: {elapsed:.0f}s | "
            f"Kecepatan: {speed:.0f} msg/s | "
            f"Estimasi sisa: {remaining:.0f}s"
            )

  except Exception as e:
    fail_count += 1
    # Hanya menampilkan detail 5 pesan kegagalan pertama agar log tidak dipenuhi error
    if fail_count <=5:
      print(f"Gagal mengirim {record.get('unique_id')}: {e}")
    elif fail_count == 6:
      print(f" ...pesan error berikutnya tidak ditampilkan...")

# Memaksa pengiriman semua pesan yang tersisa di buffer memori lokal ke broker Aiven Kafka
print("\n[Kafka] melakukan flushing buffer akhir...")
producer.flush()

# Menghitung total durasi proses streaming
duration = time.time() - start_time

print('streaming selesai')
print(f"berhasil terkirim : {success_count:,} pesan")
print(f"gagal terkirim    : {fail_count:,} pesan")
print(f"waktu total       : {duration:.1f} detik ({duration/60:.1f} menit)")


Memulai streaming 108,963 data bersih ke aiven kafka
 [  1,000 / 108,963] Waktu: 295s | Kecepatan: 3 msg/s | Estimasi sisa: 31864s
 [  2,000 / 108,963] Waktu: 588s | Kecepatan: 3 msg/s | Estimasi sisa: 31432s
Gagal mengirim KAG_3884807274: KafkaTimeoutError: Timeout after waiting for 10 secs.
Gagal mengirim KAG_3884807303: KafkaTimeoutError: Timeout after waiting for 10 secs.
 [  3,000 / 108,963] Waktu: 921s | Kecepatan: 3 msg/s | Estimasi sisa: 32534s
 [  4,000 / 108,963] Waktu: 1210s | Kecepatan: 3 msg/s | Estimasi sisa: 31742s
 [  5,000 / 108,963] Waktu: 1507s | Kecepatan: 3 msg/s | Estimasi sisa: 31331s
 [  6,000 / 108,963] Waktu: 1827s | Kecepatan: 3 msg/s | Estimasi sisa: 31344s
 [  7,000 / 108,963] Waktu: 2120s | Kecepatan: 3 msg/s | Estimasi sisa: 30887s
 [  8,000 / 108,963] Waktu: 2400s | Kecepatan: 3 msg/s | Estimasi sisa: 30284s
 [  9,000 / 108,963] Waktu: 2689s | Kecepatan: 3 msg/s | Estimasi sisa: 29863s
 [ 10,000 / 108,963] Waktu: 2966s | Kecepatan: 3 msg/s | Estimasi sis

In [40]:
# ==============================================================================
# FUNGSI CELL: Menutup koneksi KafkaProducer dengan aman (*safe shutdown*) untuk 
# melepaskan resource jaringan dan koneksi ke broker.
# ==============================================================================

# Memanggil metode close() pada objek producer untuk menghentikan proses latar belakang
producer.close()
print("koneksi kafka produser ditutup dengan aman.")


koneksi kafka produser ditutup dengan aman.


In [41]:
# ==============================================================================
# FUNGSI CELL: Memeriksa integritas jumlah data di broker Aiven Kafka menggunakan 
# KafkaConsumer dengan membandingkan offset awal dan akhir dari setiap partisi 
# dengan jumlah baris data lokal (verifikasi 100% cocok).
# ==============================================================================

# Mengimpor kembali KafkaConsumer dan TopicPartition dari pustaka kafka-python
from kafka import KafkaConsumer, TopicPartition

print("[VERIFIKASI] memeriksa data di broker aiven kafka...")

# Menginisialisasi KafkaConsumer menggunakan sertifikat SSL
consumer = KafkaConsumer(
    bootstrap_servers=KAFKA_BROKER, # Parameter bootstrap_servers: Alamat server broker Kafka (host:port)
    security_protocol="SSL",        # Parameter security_protocol: Protokol keamanan (SSL) untuk mengenkripsi koneksi
    ssl_cafile=CA_FILE,             # Parameter ssl_cafile: Sertifikat CA tepercaya
    ssl_certfile=CERT_FILE,         # Parameter ssl_certfile: Sertifikat client
    ssl_keyfile=KEY_FILE            # Parameter ssl_keyfile: Private key client
)

# Mendapatkan daftar informasi partisi dari broker untuk topik 'unified_jobs'
# Parameter pertama: Nama topik target yang akan dicek partisinya ('unified_jobs')
partitions = consumer.partitions_for_topic(TOPIC_NAME)

if partitions:
  # Membuat daftar objek TopicPartition untuk setiap ID partisi yang terdeteksi
  # Parameter TopicPartition: Menerima nama topik dan ID integer partisi
  tps = [TopicPartition(TOPIC_NAME, p) for p in partitions]

  # Dapatkan nilai offset terlama (awal) di broker untuk setiap partisi
  # Parameter beginning_offsets: List objek TopicPartition yang akan dicari offset awalnya
  beginning = consumer.beginning_offsets(tps)
  # Dapatkan nilai offset terbaru (akhir) di broker untuk setiap partisi
  # Parameter end_offsets: List objek TopicPartition yang akan dicari offset akhirnya
  ending = consumer.end_offsets(tps)

  total_messages = 0
  print(f"\n Topik: {TOPIC_NAME}")
  print(f" jumlah Partisi: {len(partitions)}")
  print(f" {'Partisi':>10} | {'Offset awal':>12} | {'Offset akhir':>12} | {'Jumlah pesan':>12}")
  print(f" {'-'*10} | {'-'*12} | {'-'*12} | {'-'*12}")

  # Melakukan iterasi di setiap partisi untuk menghitung selisih offset
  for tp in tps:
    # Selisih offset akhir dan offset awal mendefinisikan jumlah pesan di dalam partisi
    count = ending[tp] - beginning[tp]
    total_messages += count
    print(f" {tp.partition:>10} | {beginning[tp]:>12,} | {ending[tp]:>12,} | {count:>12,}")

  print(f"\n Total pesan di aiven: {total_messages:,}")
  print(f" Target yang dikirim: {len(df_clean):,}")

  # Melakukan pengecekan kecocokan data
  if total_messages == len(df_clean):
    print(f"\n Verifikasi berhasil: Jumlah data cocok 100 %")
  else:
    # Menampilkan pesan peringatan jika jumlah data di broker berbeda dengan data lokal
    print(f"perhatian: ada selisih {abs(total_messages - len(df_clean)):,} pesan.")
    print(f" Hal ini bisa terjadi jika streaming sebelumnya terputus ditengah.")
else:
  print(f" Topik '{TOPIC_NAME}' tidak ditemukan!")

# Menutup koneksi KafkaConsumer
consumer.close()


[VERIFIKASI] memeriksa data di broker aiven kafka...



 Topik: unified_jobs
 jumlah Partisi: 1
    Partisi |  Offset awal | Offset akhir | Jumlah pesan
 ---------- | ------------ | ------------ | ------------
          0 |            0 |      108,964 |      108,964

 Total pesan di aiven: 108,964
 Target yang dikirim: 108,963
perhatian: ada selisih 1 pesan.
 Hal ini bisa terjadi jika streaming sebelumnya terputus ditengah.


In [ ]:
# ==============================================================================
# FUNGSI CELL: Sel kosong (tidak berisi baris kode aktif).
# ==============================================================================


In [42]:
# ==============================================================================
# FUNGSI CELL: Menarik kembali (*consuming*) seluruh pesan data pekerjaan bersih 
# dari broker Aiven Kafka, lalu menyimpannya ke file cache lokal Streamlit 
# (cached_data.csv) agar aplikasi dashboard dapat diakses cepat secara offline.
# ==============================================================================

# Mengimpor library 'uuid' untuk menjamin keunikan ID grup konsumen Kafka
import uuid

print("[CACHE] Menarik seluruh data bersih dari aiven kafka ke file cache lokal...")
# Membuat ID grup konsumen acak agar Kafka selalu membaca data dari awal (offset terkecil)
random_group_id = f"cache-refresh-{uuid.uuid4().hex[:8]}"

# Inisialisasi konsumen Kafka (KafkaConsumer)
consumer = KafkaConsumer(
    TOPIC_NAME,                     # Parameter pertama: Topik Kafka yang akan dibaca pesannya
    bootstrap_servers=KAFKA_BROKER, # Parameter bootstrap_servers: Alamat server broker Kafka tujuan
    security_protocol="SSL",        # Parameter security_protocol: Protokol koneksi aman SSL
    ssl_cafile=CA_FILE,             # Parameter ssl_cafile: Berkas sertifikat CA
    ssl_certfile=CERT_FILE,         # Parameter ssl_certfile: Berkas sertifikat client
    ssl_keyfile=KEY_FILE,           # Parameter ssl_keyfile: Berkas private key client
    auto_offset_reset="earliest",   # Parameter auto_offset_reset: Posisi awal membaca pesan di broker ('earliest' = dari awal)
    group_id=random_group_id,       # Parameter group_id: ID grup konsumen untuk melacak offset pembacaan secara kolektif
    enable_auto_commit=False,       # Parameter enable_auto_commit: Menonaktifkan penyimpanan offset otomatis agar pembacaan read-only
    # Parameter value_deserializer: Fungsi deserializer untuk mem-parsing data byte dari Kafka kembali ke JSON/dict
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))
)

print(" Menunggu alokasi partisi...")
assigned = []
start_wait = time.time()
# Melakukan polling berkala hingga broker Kafka mengalokasikan partisi untuk grup konsumen ini
while not assigned:
  # Memanggil polling pesan dari broker Kafka
  # Parameter timeout_ms: Batas waktu maksimal memblokir proses polling jika belum ada partisi ter-assign (dalam milidetik)
  consumer.poll(timeout_ms=1000)
  assigned = list(consumer.assignment())
  # Batas tunggu alokasi partisi maksimal 20 detik
  if time.time() - start_wait > 20:
    break

if not assigned:
  print(" Gagal: Tidak mendapat alokasi partisi.")
else:
  print(f" Terhubung ke {len(assigned)} partisi.")

  # Mengambil nilai offset maksimal dari setiap partisi teralokasi
  # Parameter end_offsets: List partisi yang akan diambil offset akhirnya
  end_offsets = consumer.end_offsets(assigned)

  # List untuk mengumpulkan semua record data yang ditarik dari broker
  records = []
  poll_start = time.time()

  # Perulangan untuk menarik data dari Kafka broker hingga mencapai batas offset akhir
  while True:
    # Memeriksa apakah posisi pembacaan konsumen saat ini sudah menyamai atau melampaui offset akhir di seluruh partisi
    # Parameter position: Objek TopicPartition yang akan dicari posisi offset pembacaan terakhirnya
    all_done = all(consumer.position(tp) >= end_offsets[tp] for tp in assigned)
    if all_done:
      # Keluar dari loop jika semua data berhasil ditarik
      break

    # Melakukan penarikan data (polling) dari broker dengan timeout tunggu 3 detik
    # Parameter timeout_ms: Waktu maksimal menunggu pesan baru tiba (3000 ms) sebelum mengembalikan hasil kosong
    msg_pack = consumer.poll(timeout_ms=3000)
    if msg_pack:
      for tp, messages in msg_pack.items():
        for msg in messages:
          # Memasukkan nilai pesan ke dalam list
          records.append(msg.value)

      # Mencetak progres tarikan setiap kelipatan 5.000 data
      if len(records) % 5000 == 0:
        print(f" ... {len(records):,} pesan diterima")

    # Batas pengaman: keluar otomatis jika proses polling memakan waktu lebih dari 5 menit
    if time.time() - poll_start > 300:
      print(" Timeout tercapai.")
      break

  # Menutup koneksi konsumen Kafka
  consumer.close()

  if records:
    # Membuat DataFrame dari daftar record data yang diterima
    # Parameter pertama: List objek kamus data pekerjaan yang akan dimasukkan ke DataFrame
    df_cache = pd.DataFrame(records)

    # Menentukan lokasi penyimpanan file cache lokal untuk Dashboard Streamlit
    cache_path = "pnm_dashboard_app/cached_data.csv"
    # Menyimpan DataFrame ke file CSV lokal
    # Parameter pertama: Path tujuan penyimpanan file CSV
    # Parameter index: Boolean untuk menentukan apakah kolom indeks DataFrame ditulis ke file (False = abaikan)
    df_cache.to_csv(cache_path, index=False)

    print(f"\n Cache berhasil diperbarui")
    print(f" File   : {cache_path}")
    print(f" Baris  : {len(df_cache):,}")
    # Menampilkan ukuran file cache dalam format Megabyte (MB)
    print(f" ukuran : { os.path.getsize(cache_path) / (1024*1024):.1f} MB")
  else:
    print(" gagal: tidak ada data yang diterima dari broker.")


[CACHE] Menarik seluruh data bersih dari aiven kafka ke file cache lokal...
 Menunggu alokasi partisi...
 Terhubung ke 1 partisi.
 Timeout tercapai.

 Cache berhasil diperbarui
 File   : pnm_dashboard_app/cached_data.csv
 Baris  : 10,210
 ukuran : 18.9 MB


In [43]:
# ==============================================================================
# FUNGSI CELL: Mencetak ringkasan laporan akhir eksekusi Pipeline ETL yang sukses 
# mencakup jumlah ekstraksi, langkah transformasi cleaning, jumlah data terkirim, 
# lokasi cache, dan arahan instruksi berikutnya.
# ==============================================================================

print("PIPELINE ETL SELESAI DENGAN SUKSES!")

# Menampilkan statistik rangkuman lengkap proses ETL menggunakan multi-line string python
print(f"""
  Rangkuman Proses:
  1. Data mentah diekstrak dari 3 sumber:
     - Adzuna API       : {len(df_adz):>10,} baris
     - Kaggle LinkedIn  : {len(df_kag):>10,} baris
     - O*NET Standard   : {len(df_occ):>10,} profesi

  2. Data dibersihkan secara menyeluruh:
     - Null title/description dihapus
     - Duplikat ID dihapus
     - Spam recruiter dihapus
     - Casing diseragamkan
     - Lokasi diharmonisasi
     - Whitespace dibersihkan

  3. Data bersih dikirim ke Aiven Kafka:
     - Topik             : {TOPIC_NAME}
     - Total terkirim    : {success_count:,} baris

  4. Cache lokal Streamlit diperbarui:
     - File              : pnm_dashboard_app/cached_data.csv

  Langkah selanjutnya:
  - Jalankan dashboard Streamlit untuk melihat data bersih
  - Gunakan data bersih untuk pelatihan model Machine Learning
""")


PIPELINE ETL SELESAI DENGAN SUKSES!

  Rangkuman Proses:
  1. Data mentah diekstrak dari 3 sumber:
     - Adzuna API       :        280 baris
     - Kaggle LinkedIn  :    123,849 baris
     - O*NET Standard   :      1,016 profesi

  2. Data dibersihkan secara menyeluruh:
     - Null title/description dihapus
     - Duplikat ID dihapus
     - Spam recruiter dihapus
     - Casing diseragamkan
     - Lokasi diharmonisasi
     - Whitespace dibersihkan

  3. Data bersih dikirim ke Aiven Kafka:
     - Topik             : unified_jobs
     - Total terkirim    : 108,961 baris

  4. Cache lokal Streamlit diperbarui:
     - File              : pnm_dashboard_app/cached_data.csv

  Langkah selanjutnya:
  - Jalankan dashboard Streamlit untuk melihat data bersih
  - Gunakan data bersih untuk pelatihan model Machine Learning

